# 00 — Introduction and Environment Setup

This notebook series deploys the **Immich Smart Tagging Platform** on a 3-node Kubernetes cluster on Chameleon (KVM@TACC).

## Notebook order

| Notebook | What it does |
|----------|-------------|
| `00_setup.ipynb` | Install tools, set up credentials (this one) |
| `01_provision.ipynb` | Create lease, provision 3 VMs with Terraform |
| `02_install_k8s.ipynb` | Install Kubernetes with Kubespray (~45 min) |
| `03_deploy_services.ipynb` | Deploy Immich + MLflow + MinIO |
| `04_verify.ipynb` | Verify everything is running |
| `05_teardown.ipynb` | Destroy all resources when done |

**Run notebooks in order, top to bottom, one cell at a time.**

## Repo
`https://github.com/CRUZ773/immich-iac.git`


## Step 1: Choose your project

Run the cell below to select your Chameleon project and site.


In [ ]:
# runs in Chameleon Jupyter environment
from chi import server, context

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

## Step 2: Update your keypair

In [ ]:
# runs in Chameleon Jupyter environment
server.update_keypair()

## Step 3: Install Terraform

We install Terraform into `/work/.local/bin` so it persists across sessions.


In [ ]:
# runs in Chameleon Jupyter environment
mkdir -p /work/.local/bin
wget -q https://releases.hashicorp.com/terraform/1.14.4/terraform_1.14.4_linux_amd64.zip
unzip -o -q terraform_1.14.4_linux_amd64.zip
mv terraform /work/.local/bin
rm terraform_1.14.4_linux_amd64.zip

In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
terraform version

## Step 4: Install Ansible and Kubespray dependencies

In [ ]:
# runs in Chameleon Jupyter environment
PYTHONUSERBASE=/work/.local pip install --user ansible-core==2.16.9 ansible==9.8.0

In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
export PYTHONUSERBASE=/work/.local
ansible-playbook --version

## Step 5: Clone the IaC repository

We use `--recurse-submodules` to also pull Kubespray, which is included as a Git submodule.


In [ ]:
# runs in Chameleon Jupyter environment
git clone --recurse-submodules \
  https://github.com/CRUZ773/immich-iac.git \
  /work/immich-iac

In [ ]:
# runs in Chameleon Jupyter environment
PYTHONUSERBASE=/work/.local pip install --user \
  -r /work/immich-iac/ansible/k8s/kubespray/requirements.txt

## Step 6: Set up Chameleon credentials (clouds.yaml)

1. Open the **KVM@TACC Horizon GUI**
2. Go to **Identity → Application Credentials → Create Application Credential**
   - Name: `immich-lab`
   - Expiration: your project due date (UTC)
3. Click **Create** → **Download clouds.yaml**
4. Open the downloaded file — it contains your `application_credential_id` and `application_credential_secret`
5. Open the file browser in Jupyter (left panel), navigate to `immich-iac/tf/kvm/clouds.yaml`
6. Paste in your real `application_credential_id` and `application_credential_secret`
7. Save the file

> **DON'T commit clouds.yaml to Git.** The `.gitignore` already blocks it.

Run the cell below to confirm the file has been filled in:


In [ ]:
# runs in Chameleon Jupyter environment
# This should show your real credential ID (not the placeholder)
grep "application_credential_id" /work/immich-iac/tf/kvm/clouds.yaml

---
**Continue to `01_provision.ipynb`**
